In [1]:
import pandas as pd
df = pd.read_csv("churn.csv")

In [2]:
df.shape

(7043, 21)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [5]:
print(df["Contract"].head())

0    Month-to-month
1          One year
2    Month-to-month
3          One year
4    Month-to-month
Name: Contract, dtype: object


In [6]:
print(df["customerID"].is_unique)   # True means every ID is unique

True


In [7]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [8]:
df.drop(columns=["customerID"], inplace=True)

In [9]:
df["TotalCharges"].isna().sum()

np.int64(11)

In [10]:
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

C:\Users\Saathwik Aithal\AppData\Local\Temp\ipykernel_25092\1479199042.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [11]:
df_cat = df.copy(deep=True)

In [12]:
ohe_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling",
    "PaymentMethod"
]

In [13]:
ord_cols = ["Contract"]
contract_order = [["Month-to-month", "One year", "Two year"]]

In [14]:
cat_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [15]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])
df["Contract"] = ord_enc.fit_transform(df[["Contract"]])

In [16]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[ohe_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(ohe_cols),
    index=df.index
)

df = pd.concat([df.drop(columns=ohe_cols), encoded_df], axis=1)

In [17]:
df.shape

(7043, 44)

In [18]:
print(df["Churn"].value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [19]:
scale_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [21]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [22]:
from sklearn.model_selection import train_test_split

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    r2_score
)
from sklearn.model_selection import cross_val_score

In [24]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred,pos_label=1))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(lr, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8055358410220014
Precision: 0.6572327044025157
Recall   : 0.5588235294117647
F1 Score : 0.6040462427745664
R2 Score : 0.002645379627476907
CV Scores: [0.80269695 0.81121363 0.7920511  0.81178977 0.80539773]
Mean CV Accuracy: 0.8046298349893541

Confusion Matrix
[[926 109]
 [165 209]]


In [25]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7806955287437899
Precision: 0.6116838487972509
Recall   : 0.47593582887700536
F1 Score : 0.5353383458646617
R2 Score : -0.12475393319383077
CV Scores: [0.79630944 0.78708304 0.76366217 0.79616477 0.79829545]
Mean CV Accuracy: 0.7883029751919479

Confusion Matrix
[[922 113]
 [196 178]]


In [26]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(xgb, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7778566359119943
Precision: 0.5950155763239875
Recall   : 0.5106951871657754
F1 Score : 0.5496402877697841
R2 Score : -0.13931385465912305
CV Scores: [0.79488999 0.79701916 0.77714691 0.78480114 0.80255682]
Mean CV Accuracy: 0.7912828045357765

Confusion Matrix
[[905 130]
 [183 191]]


In [27]:
df_cat["Churn"] = df_cat["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [28]:
X_cat = df_cat.drop("Churn", axis=1)
y_cat = df_cat["Churn"]

In [29]:
from sklearn.model_selection import train_test_split

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

In [30]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=0,
    cat_features=cat_features
)

cat.fit(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features
)

CatBoostClassifier(cat_features=['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'], depth=6, iterations=500, learning_rate=0.1, random_seed=42, verbose=0)

In [32]:
y_pred = cat.predict(X_test_cat)

print("Accuracy :", accuracy_score(y_test_cat, y_pred))
print("Precision:", precision_score(y_test_cat, y_pred))
print("Recall   :", recall_score(y_test_cat, y_pred))
print("F1 Score :", f1_score(y_test_cat, y_pred))
print("R2 Score :", r2_score(y_test_cat, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_cat, y_pred))

Accuracy : 0.7998580553584103
Precision: 0.652317880794702
Recall   : 0.5267379679144385
F1 Score : 0.5828402366863905
R2 Score : -0.026474463303107765

Confusion Matrix
[[930 105]
 [177 197]]


In [33]:
from catboost import Pool, cv

train_pool = Pool(
    data=X_cat,
    label=y_cat,
    cat_features=cat_features
)

In [34]:
params = {
    "loss_function": "Logloss",
    "eval_metric": "Accuracy",
    "iterations": 500,
    "learning_rate": 0.1,
    "depth": 6,
    "random_seed": 42,
    "verbose": False
}

cv_results = cv(
    pool=train_pool,
    params=params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42
)

print(cv_results.tail())
print("Mean CV Accuracy:", cv_results["test-Accuracy-mean"].iloc[-1])

Training on fold [0/5]

bestTest = 0.8069552874
bestIteration = 40

Training on fold [1/5]

bestTest = 0.819020582
bestIteration = 41

Training on fold [2/5]

bestTest = 0.8147622427
bestIteration = 33

Training on fold [3/5]

bestTest = 0.805535841
bestIteration = 33

Training on fold [4/5]

bestTest = 0.7924662402
bestIteration = 93

     iterations  test-Accuracy-mean  test-Accuracy-std  train-Accuracy-mean  \
495         495            0.785597           0.014616             0.899297   
496         496            0.785882           0.013693             0.899475   
497         497            0.785882           0.013876             0.899439   
498         498            0.785882           0.014515             0.899475   
499         499            0.786024           0.014292             0.899830   

     train-Accuracy-std  test-Logloss-mean  test-Logloss-std  \
495            0.004266           0.442618          0.021739   
496            0.004139           0.442621          0.02156

In [35]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score

for n in [200, 400, 600, 800, 1000]:
    cat = CatBoostClassifier(
        iterations=n,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=0
    )

    cat.fit(
        X_train_cat,
        y_train_cat,
        cat_features=cat_features
    )

    y_pred = cat.predict(X_test_cat)

    print(f"\nIterations = {n}")
    print("Accuracy :", accuracy_score(y_test_cat, y_pred))
    print("Precision:", precision_score(y_test_cat, y_pred))
    print("Recall   :", recall_score(y_test_cat, y_pred))
    print("F1 Score :", f1_score(y_test_cat, y_pred))
    print("R2 Score :", r2_score(y_test_cat, y_pred))


Iterations = 200
Accuracy : 0.7998580553584103
Precision: 0.6554054054054054
Recall   : 0.5187165775401069
F1 Score : 0.5791044776119403
R2 Score : -0.026474463303107765

Iterations = 400
Accuracy : 0.7991483321504613
Precision: 0.6531986531986532
Recall   : 0.5187165775401069
F1 Score : 0.5782414307004471
R2 Score : -0.030114443669430724

Iterations = 600
Accuracy : 0.7977288857345636
Precision: 0.6488294314381271
Recall   : 0.5187165775401069
F1 Score : 0.5765230312035661
R2 Score : -0.037394404402076864

Iterations = 800
Accuracy : 0.7984386089425124
Precision: 0.6461038961038961
Recall   : 0.5320855614973262
F1 Score : 0.5835777126099707
R2 Score : -0.033754424035753905

Iterations = 1000
Accuracy : 0.7984386089425124
Precision: 0.6480263157894737
Recall   : 0.5267379679144385
F1 Score : 0.5811209439528023
R2 Score : -0.033754424035753905


In [36]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score

for d in [4, 6, 8, 10]:
    cat = CatBoostClassifier(
        iterations=400,
        learning_rate=0.1,
        depth=d,
        random_seed=42,
        verbose=0
    )

    cat.fit(
        X_train_cat,
        y_train_cat,
        cat_features=cat_features
    )

    y_pred = cat.predict(X_test_cat)

    print(f"\nDepth = {d}")
    print("Accuracy :", accuracy_score(y_test_cat, y_pred))
    print("Precision:", precision_score(y_test_cat, y_pred))
    print("Recall   :", recall_score(y_test_cat, y_pred))
    print("F1 Score :", f1_score(y_test_cat, y_pred))
    print("R2 Score :", r2_score(y_test_cat, y_pred))


Depth = 4
Accuracy : 0.8062455642299503
Precision: 0.6723549488054608
Recall   : 0.5267379679144385
F1 Score : 0.5907046476761619
R2 Score : 0.006285359993799977

Depth = 6
Accuracy : 0.7991483321504613
Precision: 0.6531986531986532
Recall   : 0.5187165775401069
F1 Score : 0.5782414307004471
R2 Score : -0.030114443669430724

Depth = 8
Accuracy : 0.7998580553584103
Precision: 0.6575342465753424
Recall   : 0.5133689839572193
F1 Score : 0.5765765765765766
R2 Score : -0.026474463303107765

Depth = 10
Accuracy : 0.7899219304471257
Precision: 0.6274509803921569
Recall   : 0.5133689839572193
F1 Score : 0.5647058823529412
R2 Score : -0.07743418843163075


In [37]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score

for lr in [0.03, 0.05, 0.07, 0.1, 0.2]:
    cat = CatBoostClassifier(
        iterations=400,
        learning_rate=lr,
        depth=4,
        random_seed=42,
        verbose=0
    )

    cat.fit(
        X_train_cat,
        y_train_cat,
        cat_features=cat_features
    )

    y_pred = cat.predict(X_test_cat)

    print(f"\nLearning Rate = {lr}")
    print("Accuracy :", accuracy_score(y_test_cat, y_pred))
    print("Precision:", precision_score(y_test_cat, y_pred))
    print("Recall   :", recall_score(y_test_cat, y_pred))
    print("F1 Score :", f1_score(y_test_cat, y_pred))
    print("R2 Score :", r2_score(y_test_cat, y_pred))


Learning Rate = 0.03
Accuracy : 0.8019872249822569
Precision: 0.6643598615916955
Recall   : 0.5133689839572193
F1 Score : 0.579185520361991
R2 Score : -0.015554522204138443

Learning Rate = 0.05
Accuracy : 0.8019872249822569
Precision: 0.6702508960573477
Recall   : 0.5
F1 Score : 0.5727411944869831
R2 Score : -0.015554522204138443

Learning Rate = 0.07
Accuracy : 0.8005677785663591
Precision: 0.6608996539792388
Recall   : 0.5106951871657754
F1 Score : 0.5761689291101055
R2 Score : -0.022834482936784584

Learning Rate = 0.1
Accuracy : 0.8062455642299503
Precision: 0.6723549488054608
Recall   : 0.5267379679144385
F1 Score : 0.5907046476761619
R2 Score : 0.006285359993799977

Learning Rate = 0.2
Accuracy : 0.8026969481902059
Precision: 0.66
Recall   : 0.5294117647058824
F1 Score : 0.5875370919881305
R2 Score : -0.011914541837815484


In [38]:
for penalty in ["l1", "l2"]:
    lr = LogisticRegression(
        penalty=penalty,
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    print(f"\nPenalty: {penalty}")
    print("Accuracy:", accuracy_score(y_test, y_pred))


Penalty: l1
Accuracy: 0.8019872249822569

Penalty: l2
Accuracy: 0.8055358410220014


In [39]:
for c in [0.01, 0.1, 1, 10, 100]:
    lr = LogisticRegression(
        C=c,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    print(f"\nC = {c}")
    print("Accuracy:", accuracy_score(y_test, y_pred))


C = 0.01
Accuracy: 0.7991483321504613

C = 0.1
Accuracy: 0.7991483321504613

C = 1
Accuracy: 0.8055358410220014

C = 10
Accuracy: 0.8048261178140526

C = 100
Accuracy: 0.8005677785663591


In [40]:
lr = LogisticRegression(
    penalty="l2",
    C=1,
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

In [41]:
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred,pos_label=1))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(lr, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7381121362668559
Precision: 0.504302925989673
Recall   : 0.7834224598930482
F1 Score : 0.6136125654450262
R2 Score : -0.3431527551732154
CV Scores: [0.75088715 0.75656494 0.73527324 0.74431818 0.74857955]
Mean CV Accuracy: 0.7471246128782502

Confusion Matrix
[[747 288]
 [ 81 293]]


In [42]:

for n in [100, 200, 300, 500]:
    rf = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print(f"\nn_estimators = {n}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


n_estimators = 100
Accuracy : 0.7828246983676366
Precision: 0.6172413793103448
Recall   : 0.4786096256684492
F1 Score : 0.5391566265060241
R2 Score : -0.11383399209486145

n_estimators = 200
Accuracy : 0.7806955287437899
Precision: 0.6116838487972509
Recall   : 0.47593582887700536
F1 Score : 0.5353383458646617
R2 Score : -0.12475393319383077

n_estimators = 300
Accuracy : 0.7842441447835344
Precision: 0.6198630136986302
Recall   : 0.4839572192513369
F1 Score : 0.5435435435435435
R2 Score : -0.1065540313622153

n_estimators = 500
Accuracy : 0.7828246983676366
Precision: 0.6164383561643836
Recall   : 0.48128342245989303
F1 Score : 0.5405405405405406
R2 Score : -0.11383399209486145


In [43]:
for d in [5, 10, 15, 20, None]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=d,
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print(f"\nmax_depth = {d}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


max_depth = 5
Accuracy : 0.794180269694819
Precision: 0.6794871794871795
Recall   : 0.42513368983957217
F1 Score : 0.5230263157894737
R2 Score : -0.055594306233692325

max_depth = 10
Accuracy : 0.8055358410220014
Precision: 0.6700680272108843
Recall   : 0.5267379679144385
F1 Score : 0.5898203592814372
R2 Score : 0.002645379627476907

max_depth = 15
Accuracy : 0.7856635911994322
Precision: 0.6216216216216216
Recall   : 0.4919786096256685
F1 Score : 0.5492537313432836
R2 Score : -0.09927407062956917

max_depth = 20
Accuracy : 0.7835344215755855
Precision: 0.6185567010309279
Recall   : 0.48128342245989303
F1 Score : 0.5413533834586466
R2 Score : -0.11019401172853849

max_depth = None
Accuracy : 0.7842441447835344
Precision: 0.6198630136986302
Recall   : 0.4839572192513369
F1 Score : 0.5435435435435435
R2 Score : -0.1065540313622153


In [44]:
for s in [2, 5, 10, 20]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=s,
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print(f"\nmin_samples_split = {s}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


min_samples_split = 2
Accuracy : 0.8055358410220014
Precision: 0.6700680272108843
Recall   : 0.5267379679144385
F1 Score : 0.5898203592814372
R2 Score : 0.002645379627476907

min_samples_split = 5
Accuracy : 0.8026969481902059
Precision: 0.6666666666666666
Recall   : 0.5133689839572193
F1 Score : 0.5800604229607251
R2 Score : -0.011914541837815484

min_samples_split = 10
Accuracy : 0.8026969481902059
Precision: 0.6655172413793103
Recall   : 0.516042780748663
F1 Score : 0.5813253012048193
R2 Score : -0.011914541837815484

min_samples_split = 20
Accuracy : 0.8048261178140526
Precision: 0.6736842105263158
Recall   : 0.5133689839572193
F1 Score : 0.582701062215478
R2 Score : -0.000994600738846163


In [46]:
for leaf in [1, 2, 4, 8]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=leaf,
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print(f"\nmin_samples_leaf = {leaf}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


min_samples_leaf = 1
Accuracy : 0.8055358410220014
Precision: 0.6700680272108843
Recall   : 0.5267379679144385
F1 Score : 0.5898203592814372
R2 Score : 0.002645379627476907

min_samples_leaf = 2
Accuracy : 0.8019872249822569
Precision: 0.6610169491525424
Recall   : 0.5213903743315508
F1 Score : 0.5829596412556054
R2 Score : -0.015554522204138443

min_samples_leaf = 4
Accuracy : 0.8090844570617459
Precision: 0.6829268292682927
Recall   : 0.5240641711229946
F1 Score : 0.5930408472012103
R2 Score : 0.02084528145909237

min_samples_leaf = 8
Accuracy : 0.8041163946061036
Precision: 0.6737588652482269
Recall   : 0.5080213903743316
F1 Score : 0.5792682926829268
R2 Score : -0.004634581105169122


In [47]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=4,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8090844570617459
Precision: 0.6829268292682927
Recall   : 0.5240641711229946
F1 Score : 0.5930408472012103
R2 Score : 0.02084528145909237
CV Scores: [0.80979418 0.80766501 0.78424414 0.81036932 0.80326705]
Mean CV Accuracy: 0.8030679398670884

Confusion Matrix
[[944  91]
 [178 196]]

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.91      0.88      1035
           1       0.68      0.52      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409



In [48]:
for n in [100, 200, 300, 500]:
    xgb = XGBClassifier(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    print(f"\nn_estimators = {n}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


n_estimators = 100
Accuracy : 0.7970191625266146
Precision: 0.6392405063291139
Recall   : 0.5401069518716578
F1 Score : 0.5855072463768116
R2 Score : -0.041034384768400045

n_estimators = 200
Accuracy : 0.7778566359119943
Precision: 0.5950155763239875
Recall   : 0.5106951871657754
F1 Score : 0.5496402877697841
R2 Score : -0.13931385465912305

n_estimators = 300
Accuracy : 0.7785663591199432
Precision: 0.5950920245398773
Recall   : 0.5187165775401069
F1 Score : 0.5542857142857143
R2 Score : -0.1356738742928001

n_estimators = 500
Accuracy : 0.7757274662881476
Precision: 0.5900621118012422
Recall   : 0.5080213903743316
F1 Score : 0.5459770114942529
R2 Score : -0.15023379575809237


In [49]:
for lr in [0.01, 0.03, 0.05, 0.1, 0.2]:
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    print(f"\nlearning_rate = {lr}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


learning_rate = 0.01
Accuracy : 0.7835344215755855
Precision: 0.7633587786259542
Recall   : 0.26737967914438504
F1 Score : 0.39603960396039606
R2 Score : -0.11019401172853849

learning_rate = 0.03
Accuracy : 0.7977288857345636
Precision: 0.6550522648083623
Recall   : 0.5026737967914439
F1 Score : 0.5688350983358548
R2 Score : -0.037394404402076864

learning_rate = 0.05
Accuracy : 0.7963094393186657
Precision: 0.6464646464646465
Recall   : 0.5133689839572193
F1 Score : 0.5722801788375559
R2 Score : -0.044674365134723004

learning_rate = 0.1
Accuracy : 0.7970191625266146
Precision: 0.6392405063291139
Recall   : 0.5401069518716578
F1 Score : 0.5855072463768116
R2 Score : -0.041034384768400045

learning_rate = 0.2
Accuracy : 0.7877927608232789
Precision: 0.6153846153846154
Recall   : 0.5347593582887701
F1 Score : 0.5722460658082976
R2 Score : -0.08835412953060007


In [50]:
for d in [3, 4, 5, 6, 8]:
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=d,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    print(f"\nmax_depth = {d}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("R2 Score :", r2_score(y_test, y_pred))


max_depth = 3
Accuracy : 0.8055358410220014
Precision: 0.6968503937007874
Recall   : 0.4732620320855615
F1 Score : 0.5636942675159236
R2 Score : 0.002645379627476907

max_depth = 4
Accuracy : 0.8069552874378992
Precision: 0.6783216783216783
Recall   : 0.5187165775401069
F1 Score : 0.5878787878787879
R2 Score : 0.009925340360123047

max_depth = 5
Accuracy : 0.801277501774308
Precision: 0.6666666666666666
Recall   : 0.5026737967914439
F1 Score : 0.573170731707317
R2 Score : -0.019194502570461625

max_depth = 6
Accuracy : 0.7977288857345636
Precision: 0.6550522648083623
Recall   : 0.5026737967914439
F1 Score : 0.5688350983358548
R2 Score : -0.037394404402076864

max_depth = 8
Accuracy : 0.7913413768630234
Precision: 0.6333333333333333
Recall   : 0.5080213903743316
F1 Score : 0.5637982195845698
R2 Score : -0.0701542276989846


In [51]:
for s in [0.6, 0.8, 1.0]:
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=4,
        subsample=s,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    print(f"\nsubsample = {s}")
    print("Accuracy :", accuracy_score(y_test, y_pred))


subsample = 0.6
Accuracy : 0.8062455642299503

subsample = 0.8
Accuracy : 0.8069552874378992

subsample = 1.0
Accuracy : 0.8069552874378992


In [54]:
for c in [0.6, 0.8, 1.0]:
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=c,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    print(f"\ncolsample_bytree = {c}")
    print("Accuracy :", accuracy_score(y_test, y_pred))


colsample_bytree = 0.6
Accuracy : 0.8055358410220014

colsample_bytree = 0.8
Accuracy : 0.801277501774308

colsample_bytree = 1.0
Accuracy : 0.8069552874378992


In [56]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=1,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(xgb, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8069552874378992
Precision: 0.6746575342465754
Recall   : 0.5267379679144385
F1 Score : 0.5915915915915916
R2 Score : 0.009925340360123047
CV Scores: [0.81476224 0.80269695 0.79134138 0.81036932 0.80184659]
Mean CV Accuracy: 0.8042032953738951

Confusion Matrix
[[940  95]
 [177 197]]

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.53      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409

